In [13]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

print("TF version:", tf.__version__)
print("Python:", os.sys.version)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(15):
        if (cur / "data" / "train").exists() and (cur / "data" / "test").exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError("Can't find project root containing data/train and data/test")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("train exists:", (DATA_DIR / "train").exists())
print("test exists:", (DATA_DIR / "test").exists())


In [ ]:
from PIL import Image

def remove_corrupted_images(root: Path) -> int:
    removed = 0
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            try:
                with Image.open(p) as img:
                    img.verify()
            except Exception:
                p.unlink(missing_ok=True)
                removed += 1
    return removed

removed_train = remove_corrupted_images(DATA_DIR / "train")
removed_test = remove_corrupted_images(DATA_DIR / "test")
print("Removed corrupted images:", removed_train + removed_test)


In [14]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
VAL_SPLIT = 0.15
EPOCHS = 10


In [15]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    str(DATA_DIR / "train"),
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset="training",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(DATA_DIR / "train"),
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset="validation",
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    str(DATA_DIR / "test"),
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


Found 557 files belonging to 2 classes.
Using 474 files for training.
Found 557 files belonging to 2 classes.
Using 83 files for validation.
Found 140 files belonging to 2 classes.
Classes: ['cats', 'dogs']


In [16]:
plt.figure(figsize=(8, 8))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.show()


NameError: name 'plt' is not defined

In [ ]:
data_aug = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
    ],
    name="data_aug",
)

In [ ]:
def build_model(num_classes: int):
    # Try transfer learning first (best accuracy)
    try:
        base = tf.keras.applications.MobileNetV2(
            input_shape=IMG_SIZE + (3,),
            include_top=False,
            weights="imagenet",
        )
        base.trainable = False
        using_imagenet = True
        print("✅ Using ImageNet weights.")
    except Exception as e:
        print("⚠️ Could not download ImageNet weights. Training from scratch.")
        print("Reason:", e)
        base = tf.keras.applications.MobileNetV2(
            input_shape=IMG_SIZE + (3,),
            include_top=False,
            weights=None,
        )
        base.trainable = True
        using_imagenet = False

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = data_aug(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)

    return model, using_imagenet

model, using_imagenet = build_model(len(class_names))
model.summary()


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2),
]


In [17]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)


NameError: name 'model' is not defined

In [18]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy:", test_acc)

NameError: name 'model' is not defined

In [ ]:
if using_imagenet:
    # Unfreeze top layers for fine-tuning
    base_model = None
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model) and layer.name.startswith("mobilenetv2"):
            base_model = layer
            break

    # Якщо base_model не знайшовся (залежить від TF), просто розморозимо всі, крім Dense head
    for l in model.layers:
        if l.name == "mobilenetv2_1.00_224":
            base_model = l

    # універсально: розморозимо все крім BatchNorm (часто стабільніше)
    for layer in model.layers:
        if "mobilenet" in layer.name.lower():
            layer.trainable = True

    # нижчий LR для fine-tuning
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    ft_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=3,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=1, restore_best_weights=True)],
    )

    test_loss, test_acc = model.evaluate(test_ds)
    print("After fine-tuning test accuracy:", test_acc)
else:
    print("Skipping fine-tuning (no ImageNet weights).")


In [ ]:
model_path = MODEL_DIR / "model.keras"
labels_path = MODEL_DIR / "labels.txt"

model.save(str(model_path))

with open(labels_path, "w", encoding="utf-8") as f:
    for name in class_names:
        f.write(name + "\n")

print("Saved:", model_path)
print("Saved:", labels_path)


In [19]:
# Take one batch from test and predict
for images, labels in test_ds.take(1):
    preds = model.predict(images, verbose=0)
    pred_ids = np.argmax(preds, axis=1)

    for i in range(min(5, images.shape[0])):
        true_name = class_names[int(labels[i])]
        pred_name = class_names[int(pred_ids[i])]
        conf = float(np.max(preds[i]))

        plt.figure(figsize=(3, 3))
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"true={true_name} | pred={pred_name} | {conf:.2f}")
        plt.axis("off")
        plt.show()
    break


NameError: name 'model' is not defined